# Breakpoints

### Boilerplate code - llm initiation

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv("./env")

google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

google_llm = ChatGoogleGenerativeAI(
    temperature=0, 
    model="gemini-2.5-flash", 
    api_key=google_api_key,
    max_tokens=200
)

openai_llm = ChatOpenAI(
    temperature=0, 
    model="gpt-5-mini", 
    api_key=openai_api_key
)

ollama_llm = ChatOllama(
    model="llama3.1:latest",
    temperature=0
)

# openai_llm.invoke("what is 2+2").content

In [ ]:
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

from langgraph.checkpoint.memory import MemorySaver

In [ ]:
memory = MemorySaver()


def add(a:int, b:int) -> int:
    "Adds two numbers - a and b"
    return a + b

def subtract(a:int, b:int) -> int:
    "Subtracts two numbers- a and b"
    return a - b

def multiply(a:int, b:int) -> int:
    "Multiplies two numbers - a and b"
    return a * b

def divide(a:int, b:int) -> int:
    "Divides two numbers - a and b"
    return a / b

tools = [add, subtract, multiply, divide]

sys_message = SystemMessage(content="You are a helpful assistant. Always use given tools for arithmetic operations")

def calculation(state: MessagesState) -> MessagesState:
    return {"messages": [llm_with_tools.invoke([sys_message] + state['messages'])]}


builder = StateGraph(MessagesState)

builder.add_node("calculation", calculation)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "calculation")
builder.add_conditional_edges("calculation", tools_condition)
builder.add_edge("tools", "calculation")

graph = builder.compile(interrupt_before=["tools"], checkpointer=memory)

llm_with_tools = openai_llm.bind_tools(tools)


display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

##### Example: Typical .invoke

In [ ]:
config={"configurable":{"thread_id": "123"}}

res = graph.invoke({"messages": [HumanMessage(content="Multiple 3 by 3, then add the result with 1")]}, config=config)
# print(res)

for msg in res['messages']:
    msg.pretty_print()

##### Stream graph with stream_mode as "values"

In [ ]:
config = {"configurable": {"thread_id": "laklaka"}}

chunks = graph.stream(
    {"messages": [HumanMessage(content="Divide 500 by 10, then add it with 50 and subtract it by 30")]},
    config=config,
    stream_mode="values"
    )

for chunk in chunks:
    chunk['messages'][-1].pretty_print()

##### Example code: .get_state(thread_id)

In [ ]:
state = graph.get_state(config)
print(state.next)

##### Example code: .get_state_history(thread_id)

In [ ]:
states = graph.get_state_history(thread)

for state in states:
    print(state)

##### Proceed to next node

In [ ]:
while graph.get_state(config).next:
    tool_name = graph.get_state(config).values['messages'][-1].tool_calls[0]['name']
    print(f"tool_name {tool_name}")

    user_approval = input(f"About to call '{tool_name}'. Approve? (yes/no): ")
    if user_approval.lower() == "yes":
        for event in graph.stream(None, config=config, stream_mode="values"):
            event['messages'][-1].pretty_print()
    else:
        print("Execution stopped by user.")
        break